# Fix Winners producers

With the initial scrap, we do not necessarily get the good details for the producers and artist columns in the file. This file aims to reconcile the data correctly

In [192]:
import pandas
import pathlib

In [193]:
path = pathlib.Path('.').joinpath('tmp/producers_to_match.json').absolute()

In [194]:
producers_to_match = pandas.read_json(path)

In [195]:
YEAR = 2018

In [196]:
df = pandas.read_csv('grammy_winners_v2.csv')

In [197]:
partial_df = df.query('year == @YEAR')

In [198]:
values_to_complete = partial_df[(partial_df.winner == True) & partial_df.producers.isna()][['id', 'category', 'song_or_album', 'artist', 'producers']]

In [199]:
values_to_complete.head(n=2)

,id,category,song_or_album,artist,producers
22144,4519,Best Pop Duo/Group Performance,Shallow,Lady Gaga & Bradley Cooper,NaN
22155,4482,Record Of The Year,This Is America,Childish Gambino,NaN


In [200]:
for item in producers_to_match.itertuples():
    matched = values_to_complete.loc[lambda x: (x.category == item.award) & (x.song_or_album == item.title), 'producers']
    if not matched.empty:
        values_to_complete.loc[matched.index, 'producers'] = item.producers

In [201]:
values_to_complete.head(n=10)

,id,category,song_or_album,artist,producers
22144,4519,Best Pop Duo/Group Performance,Shallow,Lady Gaga & Bradley Cooper,None
22155,4482,Record Of The Year,This Is America,Childish Gambino,"Donald Glover & Ludwig Göransson, producers; D..."
22161,4552,Best Rock Performance,When Bad Does Good,Chris Cornell,None
22167,4526,Best Traditional Pop Vocal Album,My Way,Willie Nelson,"Buddy Cannon & Matt Rollings, producers; Tony ..."
22172,4531,Best Pop Vocal Album,Sweetener,Ariana Grande,"Pharrell Williams, producer; Mike Larson, engi..."
22176,4567,Best Rock Album,From The Fires,Greta Van Fleet,"Herschel Boone, Al Sutton & Marlon Young, prod..."
22183,4542,Best Dance/Electronic Album,Woman Worldwide,Justice,"Justice, producers; Justice, engineers/mixers"
22188,4547,Best Contemporary Instrumental Album,Steve Gadd Band,Steve Gadd Band,"Rich Breen, engineer/mixer"
22191,4582,Best Traditional R&B Performance,How Deep Is Your Love,PJ Morton Featuring Yebba,None
22198,4557,Best Metal Performance,Electric Messiah,High On Fire,None


In [202]:
for item in values_to_complete.itertuples(name='UpdatedItem'):
    df.loc[lambda x: x.id == item.id, 'producers'] = item.producers

In [203]:
df.to_csv('grammy_winners_v2.csv', index=False)